Calibrate a camera essentially means estimating the projection matrix. A good tutorial is in https://www.youtube.com/watch?v=2XM2Rb2pfyQ
https://www.youtube.com/watch?v=GUbWsXU1mac

To calibrate a camera, we use an object of known geometry, i.e. a cube of known dimensions and with a visual pattern on it.

![object](object.png)

Hence, we know the location of every point on this objectin 3D.
We place our world coordinate frame in one of the corners of the object and we take a picture of the object.

![image](image.png)

For each point we know the world coordinates and the image coordinates.
For instance:


$P_w = [x_w, y_w, z_w]^T = [0, 3, 4]^T$

$P_I = [u, v]^T = [56, 115]^T$ in pixels

We need to (manually or semi-automatically) find several of these 3D to 2D correspondences.

For the i-th correspondence, we can write the equations:

![eq](eq.png)

which can be written as

![eq2](eq2.png)

Doing it for all the correspondences we get the following system of equations:

![eq_all](all_eq.png)

We denote with A the matrix on the left and p the vector of  the parameters. 
The system we want to solve is $Ap=0$.

Since the projection matrix can be computed up to a scale (see the book!!), we may divide by $p_{3,4}$ all the parameters or we can constrain p such that $||p||^2=1$.

In the second case, we want to find p such that Ap is as much close to 0 as possible and the norm of p is as much close as posseible to 1.
In practice, we want to solve the following minimization problem (with a very small value of $\lambda$):

$min_p ||Ap||^2 - \lambda ||p||^2$

which is called "constrained least square problem (C-LS)" and can be equivalently written as 

$min_p p^TA^TAp - \lambda p^Tp$

We solve C-LS by taking the derivative and setting it equals to 0:

$2A^TAp -2\lambda p=0$

That is $A^TAp = \lambda p$, which is the eigenvalue problem

In practice, the p we are looking for is the eigenvector corresponding to the smallest eigenvalue.

Thus, to determine the projection matrix, we compute the matrix $A^TA$ and compute its eigenvalues and eigenvectors that can be easily done by SVD decomposition.

Indeed:
If $A = USV^T$, then $A^TA=VSU^TUSV^T=VS^2V^T$. Thus $A^TAV=VS^2$
where V is the matrix of the eigenvectors and $S^2$ are the eigenvalues.

SVD can be applied by numpy.linalg.svd().




Once the projection matrix P has been estimated, we can compute the intrinsic (K) and extrinsic parameters.

Recalling that:

![projMat](projMat.png)

we will consider just part of this equation:

![subMat](subMat.png)

In practice we have the product of an upper traingular matrix and an orthogonal matrix. This reminds us of the QR factorization where Q is orthogonal and R and upper triangular matrix. 

We note that, for the property of the inverse matrix:
$(P_{1:3})^{-1} = (KR)^{-1}=R^{-1}K^{-1}$

which is an orthogonal matrix multiplicated by an upper traingular matrix. Thus, it is a QR factorization!
In practice, by applying the qr factorization (using numpy.linalg.qr) to $(P_{1:3})^{-1}$ we obtain the inverse matrix of R and K.

Once we estimate K, it is easy to estimate the translation parameters:

![translation](translation.png)

and 

$t=K^{-1}*[p_{1,4},p_{2,4},p_{3,4}]^T$

where the inverse matrix is computed by numpy. linalg. inv() .

K must be normalized properly such that K[3,3]=1. 

The openCV library provides methods to perform the camera calibration.


First, we will try to use the openCV Library

1. take one or more pictures of your checkboard (better if a cube).
In this example, we use just one image of a cube. We assume for simplicity that squares are 1 unit length and take one corner as the origin. We consider 10 correspondences: 7 corners and 3 points in the center of the three faces.
We manually estimate correspondences.

In [3]:
import numpy as np
import cv2

def define_Points():
    n_corners = 14

    #3D points
    objp = np.zeros((n_corners,3), np.float32)

    #first face
    objp[1,:] =[6, 0, 0]
    objp[2,:] =[7, 0, 7]
    objp[3,:] =[0, 0, 7]
    objp[4,:] =[3, 0, 4]

    #second face
    objp[5,:] =[0, 7, 0]
    objp[6,:] =[0, 7, 7]
    objp[7,:] =[0, 4, 4]

    #top face
    objp[8,:] =[7, 7, 7]
    objp[9,:] =[4, 4, 7]

    #additional points
    objp[10,:] =[7, 0, 3]
    objp[11,:] =[0, 3, 7]
    objp[12,:] =[0, 7, 3]
    objp[13,:] =[7, 2, 7]

    #2D points
    imgp = np.zeros((n_corners,2), np.float32)

    imgp[0,:] = [103, 254]
    imgp[1,:] = [252, 212]
    imgp[2,:] = [258, 39]
    imgp[3,:] = [103, 61]
    imgp[4,:] = [173, 132]

    imgp[5,:] = [18, 196]
    imgp[6,:] = [14, 26]
    imgp[7,:] = [50, 120]

    imgp[8,:] = [155, 10]
    imgp[9,:] = [136, 29]

    #additional points
    imgp[10,:] = [254, 141]
    imgp[11,:] = [62, 46]
    imgp[12,:] = [16, 127]
    imgp[13,:] = [224, 27]
    
    return objp, imgp

def show_points(imgp):
    # Let's show the selected points 
    img = cv2.imread("image_to_cal.png")
    img_to_show = img.copy()
    for p in imgp:
        img_to_show = cv2.circle(img_to_show, p.astype(np.int32), 3, (255, 0, 0) )

    cv2.imshow("img", img_to_show)
    cv2.waitKey(0)
    cv2.destroyAllWindows()    

def initialize_K(shape):
    # initial guess of K
    K = np.zeros((3,3))

    K[0,0]=500
    K[1,1]=500
    K[2,2]=1
    K[0,2]=shape[0]/2
    K[1,2]=shape[1]/2
    return K

def projection(objp, R, T, K, dist):
    imgpoints, _ = cv2.projectPoints(objp, R, T, K, dist)
    imgpoints = np.squeeze(imgpoints)
    return imgpoints

def draw_cube_points(K_, R_, T_, dist):
    points = []
    for x in range(8):
        for z in range(7):
            points.append([x, 0, z])

    for y in range(1, 8):
        for z in range(7):
            points.append([0, y, z])
        
    for x in range(8):
        for y in range(8):
            points.append([x, y, 7])
        
    points = np.asarray(points).astype(np.float32)

    projected_points = projection(points, R_, T_, K_, dist).astype(np.int32)
    img_to_show_res = img.copy()
    for p in projected_points:
        img_to_show_res = cv2.circle(img_to_show_res, p, 3, (255, 0, 0) )
    
    cv2.imshow("result", img_to_show_res)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [4]:
img = cv2.imread("image_to_cal.png")
objp, imgp = define_Points()
show_points(imgp)


2. compute the intrinsic (K), extrinsic (R, t) and lens deformation parameters

For planar calibration, z of the 3D points is always 0.
Our calibration is not planar. In this case, openCV expects a guess of the intrinsic matrix K 

The method is general and works also on those cases when multiple sets of correspondences are provided. This is useful when several images are used (see tutorial in the openCV library documentation). In the latter case, the method estimates 1 K matrix and several (R, t), one for each image. These pairs can identify the location of the pattern in each image in a coordinate reference system centered in the camera.

Note that R is provided as a Rodrigues rotation vector.


In [5]:
# initial guess of K
K = initialize_K(img.shape)

# calibrate
ret, K, dist, Rs, ts = cv2.calibrateCamera([objp], [imgp], img.shape[0:2], K, None, flags=cv2.CALIB_USE_INTRINSIC_GUESS|cv2.CALIB_FIX_S1_S2_S3_S4| cv2.CALIB_ZERO_TANGENT_DIST| cv2.CALIB_FIX_K2 | cv2.CALIB_FIX_K3 )

print("Intrinsic parameters")
print(K)

print("\nExtrinsic parameters")
print("R")
print(cv2.Rodrigues(Rs[0])[0])

print("T")
print(ts[0])

print("Distortion")
print(dist)


Intrinsic parameters
[[ 1.03555264e+03  0.00000000e+00  8.76338094e+01]
 [ 0.00000000e+00  9.89380467e+02 -3.08636667e+02]
 [ 0.00000000e+00  0.00000000e+00  1.00000000e+00]]

Extrinsic parameters
R
[[ 0.86462498 -0.50178478 -0.02521253]
 [ 0.04230333  0.12271347 -0.99154013]
 [ 0.50063367  0.85624379  0.1273283 ]]
T
[[ 0.66751733]
 [20.58040593]
 [35.03224517]]
Distortion
[[-0.09070622  0.          0.          0.          0.        ]]


To estimate how good are the results, we compute the mean square error of the points we manually selected and the points obtained by applying the transformation.

Finally, we use more 3D points on the visible cube faces, find projections on the image plane and visualize them.

In [6]:
imgpoints = projection(objp, Rs[0], ts[0], K, dist)

#Compute error
mean_error = (cv2.norm(imgp, imgpoints, normType=cv2.NORM_L2)**2)/len(imgpoints)
print( "total error: {}".format(mean_error) )

draw_cube_points(K, Rs[0], ts[0], dist)


total error: 17.829887433110212


We can also undistort the image removing distortions due to the lens.

We need to refine the camera matrix to control the percentage (alpha) of unwanted pixels in the undistorted image and then use the refined camera matrix to undistort the image.

In the undistorted image there may be some black pixels near the edges. Sometimes these black pixels are not desired in the final undistorted image. The method getOptimalNewCameraMatrix() returns a refined camera matrix and also the ROI (region of interest) which can be used to crop the image such that all the black pixels are excluded. 


In [7]:
img = cv2.imread('image_to_cal.png')
h,  w, _ = img.shape
alpha = 1
refined_K, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w,h), alpha, (w,h))

# undistort - Method 1
dst_1 = cv2.undistort(img, K, dist, None, None)
dst_2 = cv2.undistort(img, K, dist, None, refined_K)

#cv2.imshow("dst without refined K", dst_1)
#cv2.imshow("dst with refined K", dst_2)

#crop
x, y, w, h = roi
dst_2 = dst_2[y:y+h, x:x+w]
cv2.imshow("dst after crop", dst_2)
cv2.waitKey(1)


-1

In the second method, we use the remap function

In [8]:
img = cv2.imread('image_to_cal.png')
h,  w, _ = img.shape
alpha = 1
refined_K, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w,h), alpha, (w,h))

mapx,mapy=cv2.initUndistortRectifyMap(K,dist,None,refined_K,(w,h),5)
 
dst = cv2.remap(img,mapx,mapy,cv2.INTER_LINEAR)
 
# Displaying the undistorted image
cv2.imshow("undistorted image",dst)
cv2.waitKey(1)

-1

In [9]:
#Let's plt the ground plane on the image. 

def draw_ground(img, R, T, K, dist):
    xg = np.arange(-5, 10, 0.5) 
    yg = np.arange(-5, 10, 0.5) 
    xx, yy = np.meshgrid(xg, yg)

    dim = xx.shape[0]*xx.shape[1]
    points = np.zeros((dim,3), np.float32)

    xx = xx.reshape(dim)
    yy = yy.reshape(dim)

    points[:,0] = xx 
    points[:,1] = yy 
    points[:,2] = np.zeros((dim))

    ground, _ = cv2.projectPoints(points, R, T, K, dist)
    ground = np.squeeze(ground).astype(np.int32)

    img_to_show_res = img.copy()
    for p in ground:
        img_to_show_res = cv2.circle(img_to_show_res, p, 3, (255, 0, 0) )

    cv2.imshow("ground", img_to_show_res)
    cv2.waitKey(1)
    
draw_ground(img, Rs[0], ts[0], K, dist)

: 